# Notebook 05 — Feature Engineering

## Objectives
- Investigate the distributions of numeric features to assess whether transformations are beneficial
- Evaluate potential transformations: log, power (Yeo-Johnson), and standardisation
- Assess cardinality and encoding strategies for categorical features
- Conclude which transformations are applied in the ML pipeline

## Inputs
- `outputs/datasets/collection/telco_churn_cleaned.csv`

## Outputs
- No new files are saved — this notebook documents transformation decisions only.
- Findings feed directly into the pipeline defined in `05_ModellingEvaluation.ipynb`.

---

## 1. Change Working Directory

In [ ]:
import os

# Move up one level if running from jupyter_notebooks/
os.chdir('..') if os.path.basename(os.getcwd()) == 'jupyter_notebooks' else None
print('Working directory:', os.getcwd())

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import PowerTransformer

## 3. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_cleaned.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 4. Separate Feature Types

Split into numeric and categorical groups for targeted analysis.

In [ ]:
TARGET = 'Churn'
X = df.drop(columns=[TARGET])
y = df[TARGET]

NUMERIC_FEATURES     = X.select_dtypes(include=[np.number]).columns.tolist()
CATEGORICAL_FEATURES = X.select_dtypes(include='object').columns.tolist()

print('Numeric    :', NUMERIC_FEATURES)
print('Categorical:', CATEGORICAL_FEATURES)

## 5. Numeric Feature Distributions

We plot histograms and compute skewness to assess whether any transformation
(log, power) would help the model by making the distribution more symmetric.

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(5 * len(NUMERIC_FEATURES), 4))

for ax, col in zip(axes, NUMERIC_FEATURES):
    ax.hist(X[col], bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('Numeric Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Skewness check
skew = X[NUMERIC_FEATURES].skew().round(3)
print('\nSkewness per feature:')
print(skew)

**Finding**: `tenure` is approximately uniform (low skewness). `MonthlyCharges` is also fairly symmetric.
`TotalCharges` tends to be right-skewed because many new customers have low total charges.
We assess below whether a Yeo-Johnson power transform reduces skewness.

## 6. Assess Yeo-Johnson Power Transform

Yeo-Johnson works on positive and negative values alike. We compare skewness
before and after the transform to decide whether it adds value.

In [ ]:
pt = PowerTransformer(method='yeo-johnson', standardize=False)
X_transformed = pt.fit_transform(X[NUMERIC_FEATURES])
X_transformed_df = pd.DataFrame(X_transformed, columns=NUMERIC_FEATURES)

print('Skewness BEFORE transform:')
print(X[NUMERIC_FEATURES].skew().round(3))
print('\nSkewness AFTER Yeo-Johnson transform:')
print(X_transformed_df.skew().round(3))

**Finding**: Yeo-Johnson reduces the skewness of `TotalCharges`. However, because
tree-based ensemble methods (GradientBoostingClassifier) are invariant to monotonic
transformations — they split on thresholds, not on the absolute scale of values —
a power transform offers no benefit for our chosen classifier.

**Decision**: No power transformation is applied. `StandardScaler` is sufficient
for numerical features within the ML pipeline.

## 7. Assess Log Transform

For completeness, we check whether a log(1+x) transform reduces skewness
in TotalCharges — the most skewed feature.

In [ ]:
col = 'TotalCharges'
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(X[col], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title(f'{col} — Original (skew={X[col].skew():.2f})')

log_col = np.log1p(X[col])
axes[1].hist(log_col, bins=30, color='tomato', edgecolor='white')
axes[1].set_title(f'{col} — log(1+x) (skew={log_col.skew():.2f})')

plt.tight_layout()
plt.show()

**Finding**: Log transform reduces the right-skew of `TotalCharges`.
Again, for a tree-based classifier this has no impact on model performance.

**Decision**: Log transform is not applied.

## 8. Categorical Feature Cardinality

We check the number of unique values per categorical feature to confirm
that OneHotEncoding is appropriate and will not explode the feature space.

In [ ]:
cardinality = X[CATEGORICAL_FEATURES].nunique().sort_values(ascending=False)
print('Unique values per categorical feature:')
print(cardinality)

# Show value counts for any high-cardinality features
for col in CATEGORICAL_FEATURES:
    if cardinality[col] > 4:
        print(f'\n{col}:')
        print(X[col].value_counts())

**Finding**: All categorical features have low cardinality (6 or fewer unique values).
OneHotEncoding with `drop='first'` will produce a manageable number of dummy columns
and will not cause the curse of dimensionality.

**Decision**: `OneHotEncoder(drop='first', handle_unknown='ignore')` is appropriate.

## 9. Correlation with Target (Numeric Features)

We verify which numeric features have meaningful correlation with the Churn target,
which informs feature importance expectations from the model.

In [ ]:
corr = df[NUMERIC_FEATURES + [TARGET]].corr()['Churn'].drop('Churn').sort_values()
print('Pearson correlation with Churn:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(5, 3))
corr.plot(kind='barh', ax=ax, color=['tomato' if v > 0 else 'steelblue' for v in corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Churn', fontweight='bold')
plt.tight_layout()
plt.show()

**Finding**: `tenure` (r = -0.35) is the strongest numeric predictor — long-tenured
customers are less likely to churn. `MonthlyCharges` (r = +0.19) shows a positive
relationship. `TotalCharges` is negatively correlated but largely driven by tenure.
All three numeric features are retained.

## 10. Conclusions

| Step | Decision | Rationale |
|---|---|---|
| Power / log transform | **Not applied** | GradientBoostingClassifier is invariant to monotonic transforms |
| Numeric scaling | **StandardScaler** | Centres and scales features; required for correct SMOTE distance calculation |
| Categorical encoding | **OneHotEncoder (drop=first)** | Low cardinality; prevents multicollinearity via first-column drop |
| Feature selection | **All 19 features retained** | No near-zero variance features; all show some correlation with target |

These decisions are implemented in the `ColumnTransformer` + `ImbPipeline`
defined in `05_ModellingEvaluation.ipynb`.